# Getting Started: Gemma 4 Developer Agent

## 1. Environment Configuration and Package Installation

In [1]:
import glob
import importlib
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Configure environment variables for offline vLLM serving and LiteLLM routing
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')

# Remove broken cutlass .pth hooks if present
for pth_pattern in (
    '/usr/local/lib/python*/dist-packages/*cutlass*.pth',
    '/usr/local/lib/python*/site-packages/*cutlass*.pth',
):
    for pth in glob.glob(pth_pattern):
        try:
            os.unlink(pth)
        except OSError:
            pass

# Restore PEP 440 '+cu128' wheel filenames stripped by Kaggle dataset uploads
tmp_whl = Path('/tmp/wheelhouse')
tmp_whl.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower():
        continue
    target_name = (
        w.name.replace('cu128', '+cu128')
        if ('cu128' in w.name and '+' not in w.name)
        else w.name
    )
    target = tmp_whl / target_name
    if not target.exists():
        os.symlink(w, target)

wheels = sorted(str(w) for w in tmp_whl.glob('*.whl'))
print(f'Installing {len(wheels)} wheels from {WHEELHOUSE_DIR}...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels],
    check=True,
)
importlib.invalidate_caches()
print('Wheelhouse installation complete.')

Installing 41 wheels from /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse...
Wheelhouse installation complete.


## 2. Agent Pack and Competition Dataset

In [ ]:
import yaml
from swegemma.models import load_tasks

DATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')
WORKING_DIR = Path('/kaggle/working')
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# --- Load OUR agent pack (not sample_submission) -----------------------------
# Attach the pack as a Kaggle Dataset: either a directory containing agent.yaml,
# or a submission.zip built with agent.yaml at its ROOT (as cell 6 does).
# Set PACK_SRC explicitly if auto-discovery picks the wrong thing.
PACK_SRC = None

if PACK_SRC is None:
    hits = [
        p.parent
        for pattern in ('*/agent.yaml', '*/*/agent.yaml')
        for p in Path('/kaggle/input').glob(pattern)
        if 'sample_submission' not in p.parts
    ]
    assert len(hits) == 1, f'expected exactly one agent pack under /kaggle/input, found {hits}'
    PACK_SRC = hits[0]

AGENT_DIR = WORKING_DIR / 'submission'
if AGENT_DIR.exists():
    shutil.rmtree(AGENT_DIR)
if PACK_SRC.is_dir():
    shutil.copytree(PACK_SRC, AGENT_DIR)
else:
    AGENT_DIR.mkdir(parents=True)
    shutil.unpack_archive(str(PACK_SRC), str(AGENT_DIR))
assert (AGENT_DIR / 'agent.yaml').is_file(), (
    'agent.yaml must be at the pack root; found '
    f'{sorted(p.name for p in AGENT_DIR.iterdir())}'
)

# --- Sampling config ---------------------------------------------------------
# The pack's configs/sampling.yaml is used AS-IS. Override it here only to sweep
# sampling values without repackaging; leave None for a faithful test, since the
# real grading run reads the file shipped inside the pack (max_output_tokens
# 8192 / thinking_budget 4096 -- see configs/sampling.yaml for the derivation).
SAMPLING_OVERRIDE = None
if SAMPLING_OVERRIDE is not None:
    (AGENT_DIR / 'configs' / 'sampling.yaml').write_text(
        yaml.safe_dump(SAMPLING_OVERRIDE, sort_keys=False), encoding='utf-8'
    )
print('sampling.yaml:')
print((AGENT_DIR / 'configs' / 'sampling.yaml').read_text(encoding='utf-8'))

# --- Tasks -------------------------------------------------------------------
TASKS_PATH = DATA_DIR / 'tasks.jsonl'
tasks = load_tasks(TASKS_PATH)

print(f'Data directory: {DATA_DIR}')
print(f'Agent directory: {AGENT_DIR}  (from {PACK_SRC})')
print(f'Loaded {len(tasks)} tasks from {TASKS_PATH.name}')
print('Pack contents:')
for p in sorted(AGENT_DIR.rglob('*')):
    if p.is_file():
        print(f'  {p.relative_to(AGENT_DIR).as_posix()}')


## 3. Code Graph Functions

In [3]:
from swegemma import graph as sg

sample_task = tasks[0]
GRAPH_DIR = str(DATA_DIR / 'graphs')
EMBEDDINGS_DIR = str(DATA_DIR / 'embeddings')

# Load the pre-computed repository code graph and node embeddings for the sample task
repo_graph = sg.get_graph(
    repo_name=sample_task.repo,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
sample_nodes = list(repo_graph.nodes())
print(f'Graph for {sample_task.repo} ({sample_task.instance_id}):')
print(f'  Nodes: {repo_graph.number_of_nodes()}, Edges: {repo_graph.number_of_edges()}')

# Select a connected symbol and query structural neighbors (callers, callees, definitions)
query_node = next((n for n, deg in repo_graph.degree() if deg >= 5), sample_nodes[0])
neighbors = sg.get_neighbor(
    node=query_node,
    graph=repo_graph,
    max_neighbors=10,
)
print(f'\nNeighbors of {query_node!r} (showing up to 5):')
for n in neighbors[:5]:
    print(f'  - {n}')

# Search for semantically similar code nodes using pre-computed vector embeddings
similar = sg.get_similar_nodes(
    node=query_node,
    repo_name=sample_task.repo,
    k=5,
    graph=repo_graph,
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    base_commit=sample_task.base_commit,
)
print(f'\nTop similar nodes to {query_node!r}:')
for item in similar:
    print(f"  - {item['node_name']} (similarity={item['similarity']:.4f})")

# Extract an induced subgraph connecting the query symbol and its neighbors
focal_nodes = [query_node, *neighbors[:4]]
subgraph = sg.get_induced_subgraph(repo_graph, focal_nodes)
print(
    f'\nInduced subgraph over {len(focal_nodes)} focal nodes: '
    f'{subgraph.number_of_nodes()} nodes, {subgraph.number_of_edges()} edges'
)

Graph for fastapi/fastapi (fastapi_15661):
  Nodes: 4263, Edges: 2430

Neighbors of 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler' (showing up to 5):
  - custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute
  - fastapi.exceptions.HTTPException
  - fastapi.exceptions.RequestValidationError

Top similar nodes to 'custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute.get_route_handler':
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute.get_route_handler (similarity=1.0000)
  - fastapi.routing.APIRouter.__init__ (similarity=0.9996)
  - fastapi.applications.FastAPI.setup (similarity=0.9993)
  - custom_request_and_route.tutorial002_py310.ValidationErrorLoggingRoute (similarity=0.9991)
  - custom_request_and_route.tutorial002_an_py310.ValidationErrorLoggingRoute (similarity=0.9991)

Induced subgraph over 4 focal nodes: 4 nodes, 6 edges


## 4. Start vLLM Server

In [ ]:
import litellm
import torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
from swegemma.models.discovery import validate_single_declared_model

litellm.drop_params = True

# Pipeline test uses the small e4b weights. The pack still DECLARES
# gemma-4-31b-it-qat-w4a16-ct, and create_model_registry aliases that declared
# name to whatever MODEL_PATH is loaded -- so swapping the model for testing
# needs NO submission edit, and the pack stays submission-valid throughout.
TARGET_MODEL_NAME = 'gemma-4-e4b-it'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1')
INFERENCE_API_KEY = 'EMPTY'

# Validate single base model and discover any PEFT LoRA adapters in the submission directory
declared_model = validate_single_declared_model(AGENT_DIR)
adapters = discover_adapters(str(AGENT_DIR), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
print(f'Declared model: {declared_model}  |  LoRA adapters found: {len(adapters)}')

# e4b is ~4B params: pin TP to 1. The gpu_count heuristic would shard it 4 ways,
# wasting memory and risking attention-head divisibility errors.
# Set to None to fall back to the gpu_count heuristic.
TEST_TP_SIZE = 1
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = TEST_TP_SIZE or (4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1))

vllm_cfg = VllmConfig(
    model=str(MODEL_PATH),
    port=8000,
    host='127.0.0.1',
    tool_call_parser='gemma4',
    reasoning_parser='gemma4',
    max_model_len=32768,
    dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90,
    enable_auto_tool_choice=True,
    enable_lora=True,
    max_loras=8,
    max_lora_rank=128,
    tensor_parallel_size=tp_size,
    startup_timeout=60 * 20,
)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (model={MODEL_PATH.name}, tp={tp_size})')

# Register model and LoRA adapter aliases for Google ADK
models = server_instance.create_model_registry(
    aliases=[declared_model, TARGET_MODEL_NAME],
    model_prefix='openai/',
    api_key=INFERENCE_API_KEY,
)

## 4b. Usage and Latency Tracking

One row per LLM call (per turn), capturing context-window occupancy, thinking vs
visible output tokens, and per-turn latency. Runs across the root agent and all
sub-agents, since it hooks the shared LiteLLM call path.

In [ ]:
import time

# Per-LLM-call instrumentation. LiteLLM is already in the request path
# (`litellm.drop_params`, LITELLM_LOCAL_MODEL_COST_MAP), so one global callback
# captures every model call -- root agent and all four sub-agents alike -- with
# no change to the submission pack.
#
# One row == one turn == one LLM call. `prompt_tokens` is the size of the
# current context window at request time; `context_tokens` is its occupancy
# including this turn's generation.


class UsageTracker(litellm.CustomLogger):
    def __init__(self):
        self.rows = []

    @staticmethod
    def _get(obj, key, default=0):
        if obj is None:
            return default
        value = obj.get(key, default) if isinstance(obj, dict) else getattr(obj, key, default)
        return value if value is not None else default

    def _record(self, kwargs, response, start_time, end_time, error=None):
        usage = getattr(response, 'usage', None)
        details = self._get(usage, 'completion_tokens_details', {}) or {}
        choices = getattr(response, 'choices', None) or []
        completion = self._get(usage, 'completion_tokens', 0)
        thinking = self._get(details, 'reasoning_tokens', 0)
        prompt = self._get(usage, 'prompt_tokens', 0)
        self.rows.append({
            'call': len(self.rows) + 1,
            'started_at': start_time.isoformat(timespec='seconds'),
            'model': kwargs.get('model'),
            'prompt_tokens': prompt,          # context-window size at request time
            'output_tokens': completion,      # thinking + visible
            'thinking_tokens': thinking,
            'visible_tokens': max(completion - thinking, 0),
            'context_tokens': prompt + completion,   # window occupancy this turn
            'latency_s': round((end_time - start_time).total_seconds(), 2),
            'finish_reason': getattr(choices[0], 'finish_reason', None) if choices else None,
            'error': error,
        })

    def log_success_event(self, kwargs, response_obj, start_time, end_time):
        self._record(kwargs, response_obj, start_time, end_time)

    async def async_log_success_event(self, kwargs, response_obj, start_time, end_time):
        self._record(kwargs, response_obj, start_time, end_time)

    def log_failure_event(self, kwargs, response_obj, start_time, end_time):
        self._record(kwargs, None, start_time, end_time, error=str(response_obj)[:200])

    async def async_log_failure_event(self, kwargs, response_obj, start_time, end_time):
        self._record(kwargs, None, start_time, end_time, error=str(response_obj)[:200])


tracker = UsageTracker()
litellm.callbacks = [tracker]
# If tracker.rows stays empty after a task, this LiteLLM version routes through
# the legacy list instead -- swap the two lines (registering in BOTH double-logs
# every call, so do not set them at once):
#   litellm.callbacks = []
#   litellm.success_callback = [tracker]
print('Usage tracker registered (litellm.callbacks).')


## 5. Run Phase 1 Inference and Phase 2 Verification

In [ ]:
import asyncio
import concurrent.futures
import time
from collections import Counter

import pandas as pd
import yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator


def run_sync(coro_or_fn, *args, **kwargs):
    """Run an async coroutine synchronously inside a notebook event loop."""
    fn = (lambda: coro_or_fn(*args, **kwargs)) if callable(coro_or_fn) else (lambda: coro_or_fn)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: asyncio.run(fn())).result()
    return asyncio.run(fn())


# Read evaluation settings from the submission's eval_config.yaml
eval_config_file = AGENT_DIR / 'eval_config.yaml'
raw_eval_cfg = yaml.safe_load(eval_config_file.read_text(encoding='utf-8'))
eval_section = raw_eval_cfg.get('evaluation', raw_eval_cfg)

timeout_seconds = int(eval_section.get('timeout_seconds', 300))
max_tool_calls = int(eval_section.get('max_tool_calls', 100))
max_time_minutes = float(eval_section.get('max_time_minutes', 60.0))
turns_raw = eval_section.get('max_turns', eval_section.get('max_llm_calls'))
max_turns = int(turns_raw) if turns_raw is not None else None
print(f'Budget from {eval_config_file.name}: timeout={timeout_seconds}s, '
      f'max_tool_calls={max_tool_calls}, max_time_minutes={max_time_minutes}, max_turns={max_turns}')

# Pick a few tasks. Print the repo mix first -- choose tasks that differ in kind
# (core library fix vs docs_src/ example) rather than just taking the first N.
print('Repo mix:', Counter(t.repo for t in tasks))
SAMPLE_TASKS = tasks[:3]
print('Selected:', [t.instance_id for t in SAMPLE_TASKS])

limits, gen_constraints = build_submission_limits()

# Configure Evaluator to run Phase 1 (agent patch generation) and Phase 2 (verification)
# using the subprocess sandbox backend and the submission's eval_config parameters
eval_config = EvalConfig(
    tasks_path=TASKS_PATH,
    snapshots_dir=DATA_DIR / 'snapshots',
    results_dir=WORKING_DIR / 'results',
    submission_dir=AGENT_DIR,
    models=models,
    sandbox='subprocess',
    timeout_seconds=timeout_seconds,
    max_time_minutes=max_time_minutes,
    max_tool_calls=max_tool_calls,
    max_turns=max_turns,
    limits=limits,
    generation_constraints=gen_constraints,
    adapter_manifest=adapters,
    context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
    events_compaction_config=EventsCompactionConfig(
        compaction_interval=15,
        overlap_size=2,
        token_threshold=14336,
        event_retention_size=5,
    ),
    graph_dir=GRAPH_DIR,
    embeddings_dir=EMBEDDINGS_DIR,
    wheels_dir=DATA_DIR / 'wheels',
    verbose=False,   # True = per-event ADK logs (very noisy; useful once for per-agent detail)
)

evaluator = Evaluator(eval_config)
predictions = []
task_runs = []

for idx, task in enumerate(SAMPLE_TASKS, start=1):
    print(f'[{idx}/{len(SAMPLE_TASKS)}] Evaluating {task.instance_id} ({task.repo})...')
    start_row = len(tracker.rows)          # bound this task's tracker rows for attribution
    t0 = time.perf_counter()
    result = run_sync(
        evaluator.evaluate_task,
        task=task,
        task_index=idx,
        total_tasks=len(SAMPLE_TASKS),
    )
    wall_s = time.perf_counter() - t0
    task_runs.append({
        'task': task.instance_id,
        'repo': task.repo,
        'lo': start_row,
        'hi': len(tracker.rows),
        'wall_s': wall_s,
        'result': result,
    })
    predictions.append({'id': task.instance_id, 'prediction': result.agent_patch or ''})
    print(
        f'  -> resolved={result.resolved}, '
        f'exit_code={result.test_exit_code}, '
        f'patch_chars={len(result.agent_patch or "")}, '
        f'tool_calls={result.tool_calls}, '
        f'turns={len(tracker.rows) - start_row}, '
        f'wall={wall_s:.1f}s, harness={result.duration_seconds:.1f}s'
    )

submission_df = pd.DataFrame(predictions, columns=['id', 'prediction'])
display(submission_df)

## 5b. Token and Time Report

Per-turn table (one row per LLM call) and per-task rollup. The per-task columns
to watch:

- **turns / tool_calls** — whether the agent is pacing itself or thrashing.
- **prompt_tokens / peak_context_tokens** — context-window pressure against the
  32768 ceiling; this is what validates the `max_output_tokens` setting.
- **thinking_tokens vs visible_tokens** — how much of the output budget reasoning
  consumes before the tool call is emitted.
- **truncated_calls** — `finish_reason == "length"`; any non-zero value means the
  output budget is too small.
- **llm_time_s vs wall_s** — model time vs total including tool execution.

CSVs are written to `/kaggle/working/usage_turns.csv` and `usage_tasks.csv`.

In [ ]:
# --- Per-turn table and per-task summary ------------------------------------
# This is the baseline for prompt tuning: re-run after any prompt change and
# compare turns, tokens, and resolved on the same tasks.
turn_df = pd.DataFrame(tracker.rows)

if tracker.rows:
    summary = []
    for run in task_runs:
        calls = turn_df.iloc[run['lo']:run['hi']]
        result = run['result']
        summary.append({
            'task': run['task'],
            'turns': len(calls),
            'tool_calls': result.tool_calls,
            'prompt_tokens': int(calls.prompt_tokens.sum()),
            'output_tokens': int(calls.output_tokens.sum()),
            'thinking_tokens': int(calls.thinking_tokens.sum()),
            'visible_tokens': int(calls.visible_tokens.sum()),
            'peak_context_tokens': int(calls.context_tokens.max()) if len(calls) else 0,
            'prompt_drops': int((calls.prompt_tokens.diff() < 0).sum()),
            'truncated_calls': int((calls.finish_reason == 'length').sum()),
            'failed_calls': int(calls.error.notna().sum()),
            'llm_time_s': round(float(calls.latency_s.sum()), 1),
            'wall_s': round(run['wall_s'], 1),
            'harness_s': round(result.duration_seconds, 1),
            'resolved': result.resolved,
            'patch_chars': len(result.agent_patch or ''),
        })

    summary_df = pd.DataFrame(summary)
    display(summary_df)
    display(turn_df)

    summary_df.to_csv(WORKING_DIR / 'usage_tasks.csv', index=False)
    turn_df.to_csv(WORKING_DIR / 'usage_turns.csv', index=False)

    print(f'Peak context across all calls: {int(turn_df.context_tokens.max())} / 32768 tokens')
    print(f'Truncated calls (finish_reason=length): {int((turn_df.finish_reason == "length").sum())}')
    print('Verdict on max_output_tokens: if truncated_calls > 0 while peak context is under '
          '~24k, raise max_output_tokens toward 10240; if peak context exceeds ~28k, lower it.')
    print('prompt_drops is an upper bound on compactions: a fresh sub-agent call also has a '
          'small prompt, so a drop does not prove a compaction happened.')
else:
    summary_df = pd.DataFrame()
    print('WARNING: the tracker captured no LLM calls, so no token/time data is available. '
          'The LiteLLM callback path is not firing -- see the fallback note in the tracker cell.')

In [ ]:
# Optional cross-check: vLLM's own Prometheus counters should roughly agree with
# the tracker totals. A large divergence means the LiteLLM callback is missing a
# call path (e.g. a sub-agent using a different transport).
import requests

try:
    metrics_text = requests.get('http://127.0.0.1:8000/metrics', timeout=10).text
except Exception as exc:  # noqa: BLE001 -- report it, do not fail the run
    metrics_text = ''
    print(f'Could not read /metrics: {exc}')

# Metric names vary by vLLM version; print whatever matches, or say it is absent.
for name in (
    'vllm:prompt_tokens_total',
    'vllm:generation_tokens_total',
    'vllm:request_success_total',
):
    lines = [ln for ln in metrics_text.splitlines() if ln.startswith(name)]
    print('\n'.join(lines) if lines else f'{name}: not present in /metrics')


## 6. Package Submission Archive

In [6]:
import zipfile
from swegemma.config import ALLOWED_SUBMISSION_EXTENSIONS, MAX_SUBMISSION_SIZE_BYTES

# Create submission.zip with agent.yaml at the root of the archive
zip_base = WORKING_DIR / 'submission'
zip_path = Path(shutil.make_archive(str(zip_base), 'zip', root_dir=AGENT_DIR))

# Validate archive contents against competition constraints
with zipfile.ZipFile(zip_path, 'r') as zf:
    infos = zf.infolist()
    total_size = sum(i.file_size for i in infos)
    assert total_size <= MAX_SUBMISSION_SIZE_BYTES, f'Archive exceeds 3 GiB limit: {total_size}'
    for info in infos:
        if not info.is_dir():
            ext = Path(info.filename).suffix.lower()
            assert ext in ALLOWED_SUBMISSION_EXTENSIONS, f'Disallowed file extension: {info.filename}'

print(f'Created {zip_path} ({zip_path.stat().st_size / (1024 * 1024):.2f} MiB)')

Created /kaggle/working/submission.zip (0.17 MiB)
